In [1]:
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered_grounded import RouteMeanderGrounded
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal import MetalGUI

In [2]:
# resonator parameters
TOTAL_LENGTH_MM = 5.92905
GROUND_WIDTH_UM = 10
GROUND_OVERLAP_UM = 4
EDGE_MARGIN_MM = 0.5
GROUND_POSS_MM = 2 * TOTAL_LENGTH_MM / 3

# chip geometry (make it as tight as possible)
CHIP_SIZE_X_MM = 3.0
CHIP_SIZE_Y_MM = 2.0
CHIP_SIZE_Z_UM = 500
CHIP_CENTER_X_MM = 0.0
CHIP_CENTER_Y_MM = -0.381

In [3]:
chip_size_x_mm = CHIP_SIZE_X_MM
chip_size_y_mm = CHIP_SIZE_Y_MM
chip_center_x_mm = CHIP_CENTER_X_MM
chip_center_y_mm = CHIP_CENTER_Y_MM
ground_overlap_um = GROUND_OVERLAP_UM

ground_pos_mm = GROUND_POSS_MM
total_length_mm = TOTAL_LENGTH_MM

design = designs.DesignPlanar({}, overwrite_enabled=True)

design.chips.main.size.size_x = f'{chip_size_x_mm}mm'
design.chips.main.size.size_y = f'{chip_size_y_mm}mm'
design.chips.main.size.size_z = f'{CHIP_SIZE_Z_UM}um'
design.chips.main.size.center_x = f'{chip_center_x_mm}mm'
design.chips.main.size.center_y = f'{chip_center_y_mm}mm'

design.variables['cpw_width'] = '20 um'
design.variables['cpw_gap'] = '12.25 um'
gui = MetalGUI(design)

In [4]:
otg1 = OpenToGround(design, 'otg1', options=dict(
    chip='main', pos_x='-0.5mm', pos_y='-10um', orientation=180,
    width='20um', gap='12.25um', termination_gap='12.25um'))
# SHORT end (x=0). No termination_gap here - the conductor is tied to
# ground, so there is no open-end gap to size.
sg1 = ShortToGround(design, 'sg1', options=dict(
    chip='main', pos_x='0mm', pos_y='-0.91mm', orientation=-90,
    width='20um', gap='12.25um'))

common_kwargs = dict(
    trace_width='20um',
    trace_gap='12.25um',
    total_length=f'{total_length_mm}mm',
    hfss_wire_bonds=True,
    fillet='99.9 um',
    lead=dict(start_straight='100um', end_straight = "20um"),
    pin_inputs=Dict(
        start_pin=Dict(component='sg1', pin='short'),
        end_pin=Dict(component='otg1', pin='open')),
)
try:
    res_1.delete()
except NameError : pass

if ground_pos_mm is None:
    # Baseline: no grounding strap at all - plain quarter-wave resonator
    res1 = RouteMeander(design, 'resonator1', Dict(**common_kwargs))
else:
    if not (EDGE_MARGIN_MM <= ground_pos_mm <= total_length_mm - EDGE_MARGIN_MM):
        raise ValueError(
            f"ground_pos_mm={ground_pos_mm} out of valid range "
            f"[{EDGE_MARGIN_MM}, {total_length_mm - EDGE_MARGIN_MM}]"
        )
    res1 = RouteMeanderGrounded(design, 'resonator1', Dict(
        **common_kwargs,
        ground_straps=dict(
            positions=[f'{ground_pos_mm}mm'],
            width=f'{GROUND_WIDTH_UM}um',
            overlap=f'{ground_overlap_um}um'),
    ))
gui.rebuild()
gui.autoscale()

In [5]:
fillet='99.99um'

cpw_options = Dict(
    lead=Dict(
        start_straight='100um',
        end_straight='100um'),
    fillet=fillet
    )

def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component=pin1_comp_name,
                pin=pin1_comp_pin),
            end_pin=Dict(
                component=pin2_comp_name,
                pin=pin2_comp_pin)),
        total_length=length)
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

In [6]:
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
main_branch_height_um= 50
main_branch_half_width_um = 800
pad_1_options = dict(
        pos_x = f"-{main_branch_half_width_um}um",
        pos_y = f"{main_branch_height_um}um",
        orientation = "0"
    )

pad_2_options = dict(
        pos_x = f"{main_branch_half_width_um}um",
        pos_y = f"{main_branch_height_um}um",
        orientation = "180"
    )

try:
    pad_1.delete()
    pad_2.delete()
except NameError : pass

pad_1 = LaunchpadWirebond(
    design, 
    "pad_1",
    options = pad_1_options
)
pad_2 = LaunchpadWirebond(
    design, 
    "pad_2",
    options = pad_2_options
)
gui.rebuild()
gui.autoscale()

In [7]:
try:
    main_branch.delete()
except NameError: pass

main_branch_options = {
    "pin_inputs": {
        "start_pin": {
            "component": "pad_1", 
            "pin": "tie"
            },
        "end_pin": {
            "component": "pad_2", 
            "pin": "tie"
            }
    }
}

main_branch = RouteStraight(design, "main_branch", main_branch_options)
gui.rebuild()
gui.autoscale()

In [8]:
import geopandas as gpd
import pandas as pd

all_bounds = []
for table_name, table in design.qgeometry.tables.items():
    if len(table) == 0:
        continue
    b = table.total_bounds  # [minx, miny, maxx, maxy], in mm
    print(f"{table_name:>8}: X=[{b[0]:.4f},{b[2]:.4f}]  Y=[{b[1]:.4f},{b[3]:.4f}]")
    all_bounds.append(b)

import numpy as np
all_bounds = np.array(all_bounds)
print(f"\nOverall: X=[{all_bounds[:,0].min():.4f},{all_bounds[:,2].max():.4f}]  "
      f"Y=[{all_bounds[:,1].min():.4f},{all_bounds[:,3].max():.4f}]")

    path: X=[-0.7750,0.7750]  Y=[-0.9100,0.0500]
    poly: X=[-1.0600,1.0600]  Y=[-0.2362,0.1480]

Overall: X=[-1.0600,1.0600]  Y=[-0.9100,0.1480]
